# 04 · lite-TFT

20,173-parameter temporal fusion transformer variant: variable-selection GRN, LSTM
encoder, multi-head attention, GRN head. Causal features. Answers R1-7: dropout 0.1,
weight decay 1e-5 and early stopping on one *training* cell per fold (patience 12,
<= 120 epochs). Set `FIXED_EPOCHS = 60` to reproduce the submitted fixed-epoch protocol.

In [1]:
%run ./00_common.ipynb

root: <repository root>


In [2]:
FIXED_EPOCHS = 0          # 0 = early stopping (paper); 60 = submitted protocol
LOOK, MAXEP, PATIENCE = 10, 120, 12
SUFFIX = f"_fixed{FIXED_EPOCHS}" if FIXED_EPOCHS else ""

import torch
import torch.nn as nn
torch.set_num_threads(4)

In [3]:
class GRN(nn.Module):
    def __init__(s, di, dh, do=None, p=0.1):
        super().__init__(); do = do or di
        s.skip = nn.Linear(di, do) if di != do else nn.Identity()
        s.fc1 = nn.Linear(di, dh); s.fc2 = nn.Linear(dh, do); s.gate = nn.Linear(dh, do)
        s.drop = nn.Dropout(p); s.norm = nn.LayerNorm(do); s.act = nn.ELU()

    def forward(s, x):
        h = s.drop(s.act(s.fc1(x)))
        return s.norm(s.fc2(h) * torch.sigmoid(s.gate(h)) + s.skip(x))


class VSN(nn.Module):
    def __init__(s, nf, dm):
        super().__init__(); s.feat_grn = GRN(1, dm, dm); s.weight_grn = GRN(nf, dm, nf)

    def forward(s, x):
        Bz, T, F = x.shape
        flat = x.reshape(Bz * T, F)
        w = torch.softmax(s.weight_grn(flat), dim=-1)
        pf = s.feat_grn(flat.reshape(Bz * T * F, 1)).view(Bz * T, F, -1)
        return (pf * w.unsqueeze(-1)).sum(1).view(Bz, T, -1), w.view(Bz, T, F)


class LiteTFT(nn.Module):
    def __init__(s, nf, dm=32, nh=4):
        super().__init__()
        s.vsn = VSN(nf, dm); s.lstm = nn.LSTM(dm, dm, batch_first=True)
        s.attn = nn.MultiheadAttention(dm, nh, batch_first=True, dropout=0.1)
        s.grn_out = GRN(dm, dm); s.head = nn.Linear(dm, 1)

    def forward(s, x):
        v, w = s.vsn(x); e, _ = s.lstm(v); a, _ = s.attn(e, e, e)
        return s.head(s.grn_out(a + e)[:, -1]).squeeze(-1), w

In [4]:
def make_seq(g, cols, mu, sd):
    x = (g[cols].to_numpy(float) - mu) / sd
    rul, cyc = g["rul"].to_numpy(float), g["cycle"].to_numpy(float)
    X, y, k = [], [], []
    for i in range(len(x) - LOOK):
        X.append(x[i:i + LOOK]); y.append(rul[i + LOOK]); k.append(cyc[i + LOOK])
    return np.asarray(X, np.float32), np.asarray(y, np.float32), np.asarray(k)


def seqs(d, cols, mu, sd):
    X, y = [], []
    for _, g in d.groupby("battery_id"):
        a, b, _ = make_seq(g.sort_values("cycle"), cols, mu, sd)
        if len(a): X.append(a); y.append(b)
    return torch.tensor(np.concatenate(X)), torch.tensor(np.concatenate(y)) / 100.0


def run_tft(df, cols, batteries, tag):
    preds, vsn, log = {}, [], []
    for held in batteries:
        torch.manual_seed(SEED); np.random.seed(SEED)
        train, test = df[df.battery_id != held], df[df.battery_id == held]
        med = train[cols].median(numeric_only=True)
        train = train.fillna({c: med[c] for c in cols}); test = test.fillna({c: med[c] for c in cols})
        rng = np.random.default_rng(SEED + int(held[1:]))
        val_cell = rng.choice(sorted(train["battery_id"].unique()))
        fit_df, val_df = train[train.battery_id != val_cell], train[train.battery_id == val_cell]
        if FIXED_EPOCHS:
            fit_df, val_cell = train, "-"
        mu = fit_df[cols].to_numpy(float).mean(0); sd = fit_df[cols].to_numpy(float).std(0); sd[sd == 0] = 1
        Xf, yf = seqs(fit_df, cols, mu, sd)
        Xv, yv = seqs(val_df if not FIXED_EPOCHS else fit_df, cols, mu, sd)
        Xte, yte, kte = make_seq(test.sort_values("cycle"), cols, mu, sd)
        model = LiteTFT(len(cols))
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        crit = nn.HuberLoss()
        loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xf, yf), batch_size=32, shuffle=True)
        best, best_ep, best_state, wait = np.inf, 0, None, 0
        for ep in range(1, (FIXED_EPOCHS or MAXEP) + 1):
            model.train()
            for xb, yb in loader:
                opt.zero_grad(); p, _ = model(xb); crit(p, yb).backward(); opt.step()
            model.eval()
            with torch.no_grad():
                vl = float(torch.mean(torch.abs(model(Xv)[0] - yv)) * 100)
            if FIXED_EPOCHS:
                best, best_ep, best_state = vl, ep, {k: v.clone() for k, v in model.state_dict().items()}
                continue
            if vl < best - 1e-4:
                best, best_ep, wait = vl, ep, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= PATIENCE:
                    break
        model.load_state_dict(best_state); model.eval()
        with torch.no_grad():
            p, w = model(torch.tensor(Xte))
        preds[held] = dict(cycle=kte, rul_pred=np.clip(p.numpy() * 100, 0, None), rul_true=yte)
        vsn.append(w.mean(dim=(0, 1)).numpy())
        log.append(dict(config=tag, held_out=held, val_cell=val_cell, n_train_seq=len(Xf),
                        best_epoch=best_ep, stopped_epoch=ep, val_MAE=best))
        print(f"[{tag}] {held}: val={val_cell} best_ep={best_ep} stop={ep} n_train={len(Xf)}")
    return preds, np.mean(vsn, 0), pd.DataFrame(log)

In [5]:
cells = load_cells()
scr = screen(cells, save=False)
elig = scr[scr["eligible"]]["battery"].tolist()
df_t, cols = feature_table(cells, elig, "trailing")
allm, logs = [], []
for tag, cc in (("trailing" + SUFFIX, cols), ("trailing_noImpedance" + SUFFIX, [c for c in cols if c not in IMPEDANCE])):
    preds, vsn, log = run_tft(df_t, cc, elig, tag)
    logs.append(log)
    e = per_cell_errors({"lite-TFT": preds}, cells); e["config"] = tag; e["seed"] = SEED
    allm.append(e)
    pd.DataFrame({"feature": cc, "vsn_weight": vsn}).sort_values("vsn_weight", ascending=False) \
        .to_csv(RESULTS / f"tft_vsn_{tag}.csv", index=False)
errs = pd.concat(allm, ignore_index=True)
errs.to_csv(RESULTS / f"tft_per_cell_errors{SUFFIX}.csv", index=False)
pd.concat(logs).to_csv(RESULTS / f"tft_training_log{SUFFIX}.csv", index=False)
summ = pd.concat([summarise(errs[errs.config == c], c) for c in errs.config.unique()])
summ.to_csv(RESULTS / f"tft_summary{SUFFIX}.csv", index=False)
summ[["config", "lambda", "MAE", "MAE_sd", "MAE_ci_lo", "MAE_ci_hi", "RMSE", "HR", "alpha_acc"]].round(2)

[trailing] B0005: val=B0006 best_ep=1 stop=13 n_train=511


[trailing] B0006: val=B0007 best_ep=3 stop=15 n_train=490


[trailing] B0007: val=B0005 best_ep=24 stop=36 n_train=448


[trailing] B0018: val=B0006 best_ep=1 stop=13 n_train=539


[trailing] B0036: val=B0044 best_ep=12 stop=24 n_train=432


[trailing] B0042: val=B0043 best_ep=11 stop=23 n_train=556


... (11 outputs left out)

[trailing_noImpedance] B0044: val=B0055 best_ep=8 stop=20 n_train=566


[trailing_noImpedance] B0054: val=B0044 best_ep=32 stop=44 n_train=566


[trailing_noImpedance] B0055: val=B0006 best_ep=12 stop=24 n_train=559


,config,lambda,MAE,MAE_sd,MAE_ci_lo,MAE_ci_hi,RMSE,HR,alpha_acc
0,trailing,0.5,19.040001,18.879999,9.32,30.80,26.139999,0.4,0.4
1,trailing,0.7,13.250000,13.090000,6.10,21.39,18.160000,0.6,0.3
2,trailing,0.9,14.480000,10.310000,9.10,21.28,17.480000,0.2,0.0
0,trailing_noImpedance,0.5,25.430000,24.709999,12.62,41.42,34.590000,0.2,0.2
1,trailing_noImpedance,0.7,21.870001,18.889999,11.67,33.78,28.270000,0.3,0.1
2,trailing_noImpedance,0.9,13.150000,16.889999,4.84,24.36,20.719999,0.6,0.1


### Size, int8 size, activation memory and latency (R1-9)

In [6]:
m = LiteTFT(len(cols)).eval()
n_par = sum(p.numel() for p in m.parameters())
q = torch.ao.quantization.quantize_dynamic(m, {nn.Linear, nn.LSTM}, dtype=torch.qint8)
sz = lambda mod: (lambda b: (torch.save(mod.state_dict(), b), b.getbuffer().nbytes / 1024)[1])(io.BytesIO())
sizes = []
hooks = [mm.register_forward_hook(lambda mm, i, o: sizes.append(
    sum(t.numel() for t in (o if isinstance(o, tuple) else (o,)) if torch.is_tensor(t))))
    for mm in m.modules() if len(list(mm.children())) == 0 or isinstance(mm, (nn.LSTM, nn.MultiheadAttention))]
xb = torch.randn(1, LOOK, len(cols))
with torch.no_grad():
    m(xb)
for h in hooks:
    h.remove()
torch.set_num_threads(1)
with torch.no_grad():
    for _ in range(20): m(xb)
    L = []
    for _ in range(3):
        t = time.perf_counter()
        for _ in range(300): m(xb)
        L.append((time.perf_counter() - t) / 300 * 1e6)
dep = pd.DataFrame([dict(n_params=n_par, fp32_param_kb=round(n_par * 4 / 1024, 1),
                         state_dict_int8_kb=round(sz(q), 1), latency_us_1thread=round(sorted(L)[1], 1),
                         activation_sum_bytes=4 * sum(sizes), activation_max_tensor_bytes=4 * max(sizes))])
dep.to_csv(RESULTS / f"tft_deploy{SUFFIX}.csv", index=False)
dep

,n_params,fp32_param_kb,state_dict_int8_kb,latency_us_1thread,activation_sum_bytes,activation_max_tensor_bytes
0,20173,78.8,53.9,1344.5,189044,24320
